# 07 — Expected Loss Calculation and Portfolio Analysis

This notebook computes the Expected Loss (EL) for every loan in the portfolio and validates the results through economic coherence checks and segment-level analysis.

$$
\text{EL} = PD \times \frac{LGD}{100} \times EAD
$$

**Input:** `data/processed/checkpoint_df_master_with_PD_LGD.pkl`  
**Output:** `data/processed/portfolio_with_EL.pkl`

## Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle
import os

## 1. Load Checkpoint

In [ ]:
with open("data/processed/checkpoint_df_master_with_PD_LGD.pkl", "rb") as f:
    data = pickle.load(f)
df_master_features = data["df_master_features"]

print(f"Portfolio shape: {df_master_features.shape}")
print(f"Columns with model outputs: {[c for c in df_master_features.columns if c in ['PD', 'LGD', 'EAD']]}")

## 2. Expected Loss Calculation

LGD is divided by 100 to convert from percentage scale (0-100) to proportion (0-1), consistent with `Target_Variable_Loss = 100 * max_bal_owed / EAD`.

In [ ]:
df_master_features["EL"] = (
    df_master_features["PD"] *
    (df_master_features["LGD"] / 100) *
    df_master_features["EAD"]
)

print(f"EL computed for {len(df_master_features):,} loans")
print(f"Total EL       : {df_master_features['EL'].sum():,.0f}")
print(f"Mean EL / loan : {df_master_features['EL'].mean():,.2f}")

## 3. Validation

### 3.1 EL vs Observed Loss

In [ ]:
# Observed loss on defaulted loans — using realized LGD and EAD
mask_default = df_master_features["Target_Variable_Default"] == 1

observed_loss = (
    df_master_features.loc[mask_default, "EAD"] *
    df_master_features.loc[mask_default, "Target_Variable_Loss"] / 100
).sum()

expected_loss = df_master_features["EL"].sum()

print(f"Observed loss (defaulted loans) : {observed_loss:,.0f}")
print(f"Expected Loss (full portfolio)  : {expected_loss:,.0f}")
print(f"EL / Observed loss ratio        : {expected_loss / observed_loss:.4f}")

### 3.2 EL Decomposition by Population

In [ ]:
el_default    = df_master_features.loc[mask_default,  "EL"].sum()
el_no_default = df_master_features.loc[~mask_default, "EL"].sum()

print(f"EL defaulted loans    : {el_default:,.0f}")
print(f"EL non-defaulted loans: {el_no_default:,.0f}")
print(f"% EL from non-defaulted: {el_no_default / expected_loss * 100:.2f}%")

### 3.3 EL Rate

In [ ]:
total_ead = df_master_features["EAD"].sum()
el_rate   = expected_loss / total_ead * 100

print(f"Total EAD (portfolio): {total_ead:,.0f}")
print(f"EL rate              : {el_rate:.2f}%")
print("Expected range for US unsecured retail loans: 3-5%")

### 3.4 EL Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(df_master_features["EL"], bins=100, color="#185FA5", edgecolor="white", linewidth=0.3)
ax.set_xlabel("Expected Loss per loan", fontsize=11)
ax.set_ylabel("Count", fontsize=11)
ax.set_title("Distribution of Expected Loss — full portfolio", fontsize=12)
ax.grid(True, alpha=0.2, axis="y")
plt.tight_layout()
plt.show()

## 4. Portfolio Analysis by Segment

### 4.1 EL Rate by Loan Term

In [ ]:
el_by_term = df_master_features.groupby("loan_term_months").apply(
    lambda x: (x["EL"].sum() / x["EAD"].sum() * 100).round(2)
)
print(el_by_term)

# 60-month loans should show higher EL rate than 36-month loans
assert el_by_term[60] > el_by_term[36], "EL rate ordering violated: 60m should exceed 36m"

### 4.2 EL Rate by Interest Rate Quartile

In [ ]:
df_master_features["interest_rate_q"] = pd.qcut(
    df_master_features["interest_rate"], q=4, labels=["Q1", "Q2", "Q3", "Q4"]
)

el_by_rate = df_master_features.groupby("interest_rate_q").apply(
    lambda x: (x["EL"].sum() / x["EAD"].sum() * 100).round(2)
)
print(el_by_rate)

# Q4 (highest rates) must have higher EL rate than Q1 (lowest rates)
assert el_by_rate["Q4"] > el_by_rate["Q1"], "EL rate ordering violated: Q4 should exceed Q1"

In [ ]:
# Visualize EL rate by interest rate quartile
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(el_by_rate.index, el_by_rate.values, color="#185FA5", edgecolor="white")
ax.set_xlabel("Interest Rate Quartile", fontsize=11)
ax.set_ylabel("EL Rate (%)", fontsize=11)
ax.set_title("EL Rate by Interest Rate Quartile", fontsize=12)
ax.grid(True, alpha=0.2, axis="y")
plt.tight_layout()
plt.show()

### 4.3 PD Calibration Check by Segment

In [ ]:
# PD mean vs observed default rate per interest rate quartile
calibration_check = df_master_features.groupby("interest_rate_q").agg(
    pd_mean=("PD", "mean"),
    default_rate=("Target_Variable_Default", "mean")
).round(4)

print(calibration_check)

## 5. Save Final Portfolio

In [ ]:
os.makedirs("data/processed", exist_ok=True)

with open("data/processed/portfolio_with_EL.pkl", "wb") as f:
    pickle.dump({"df_master_features": df_master_features}, f)

print(f"Final portfolio saved. Shape: {df_master_features.shape}")
print(f"Columns: {df_master_features.columns.tolist()}")